# Tien xu ly du lieu Credit Card truoc khi dung Local Outlier Factor (LOF)


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

DATA_PATH = Path('creditcard.csv')
OUTPUT_PATH = Path('creditcard_preprocessed_for_lof.csv')
TARGET_COL = 'Class'
RANDOM_STATE = 42
N_NORMAL_SAMPLE = 10_000

pd.set_option('display.max_columns', 40)

## 1. Doc va khao sat du lieu ban dau

`Class` la nhan that (`0`: binh thuong, `1`: gian lan). Cot nay duoc giu lai de lay mau va danh gia ket qua, nhung se khong duoc dua vao cac dac trung dau vao cua LOF.

In [10]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay file: {DATA_PATH.resolve()}')

df_raw = pd.read_csv(DATA_PATH)
if TARGET_COL not in df_raw.columns:
    raise ValueError(f'Du lieu can co cot nhan {TARGET_COL!r}.')

print(f'Kich thuoc ban dau: {df_raw.shape[0]:,} dong x {df_raw.shape[1]} cot')
print('\nPhan bo nhan Class:')
display(df_raw[TARGET_COL].value_counts(dropna=False).rename('so_luong').to_frame())
display(df_raw.head())

Kich thuoc ban dau: 284,807 dong x 31 cot

Phan bo nhan Class:


,so_luong
0,284315
1,492


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [11]:
summary_initial = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'missing_%': (df_raw.isna().mean() * 100).round(4),
    'unique': df_raw.nunique(dropna=False)
})

numeric_raw = df_raw.select_dtypes(include=np.number)
non_finite_total = int((~np.isfinite(numeric_raw)).sum().sum())
duplicate_total = int(df_raw.duplicated().sum())

print(f'Tong gia tri missing: {int(df_raw.isna().sum().sum()):,}')
print(f'Tong gia tri vo han/khong huu han: {non_finite_total:,}')
print(f'So dong trung lap: {duplicate_total:,}')
display(summary_initial)

Tong gia tri missing: 0
Tong gia tri vo han/khong huu han: 0
So dong trung lap: 1,081


,dtype,missing,missing_%,unique
Time,float64,0,0.0,124592
V1,float64,0,0.0,275663
V2,float64,0,0.0,275663
V3,float64,0,0.0,275663
V4,float64,0,0.0,275663
V5,float64,0,0.0,275663
V6,float64,0,0.0,275663
V7,float64,0,0.0,275663
V8,float64,0,0.0,275663
V9,float64,0,0.0,275663


## 2. Xu ly missing values va du lieu khong hop le

- Thay `+inf` va `-inf` thanh `NaN` de xu ly thong nhat.
- `Amount < 0` khong hop le ve nghiep vu, nen duoc coi la missing neu xuat hien.
- Dong thieu nhan `Class` bi loai vi khong the dung de danh gia.
- Cac dac trung so thieu du lieu duoc dien bang **trung vi** (median), it bi anh huong boi giao dich co gia tri rat lon.

In [12]:
df = df_raw.copy()
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

if 'Amount' in df.columns:
    invalid_amount_count = int((df['Amount'] < 0).sum())
    df.loc[df['Amount'] < 0, 'Amount'] = np.nan
else:
    invalid_amount_count = 0

missing_target = int(df[TARGET_COL].isna().sum())
df = df.dropna(subset=[TARGET_COL]).copy()

feature_cols = [col for col in df.columns if col != TARGET_COL]
non_numeric_features = df[feature_cols].select_dtypes(exclude=np.number).columns.tolist()
if non_numeric_features:
    raise TypeError(f'LOF can cac dac trung so; can ma hoa them cac cot: {non_numeric_features}')

missing_features_before = int(df[feature_cols].isna().sum().sum())
imputer = SimpleImputer(strategy='median')
df[feature_cols] = imputer.fit_transform(df[feature_cols])

print(f'So Amount am duoc coi la missing: {invalid_amount_count:,}')
print(f'So dong bi loai vi thieu Class: {missing_target:,}')
print(f'So gia tri dac trung duoc dien median: {missing_features_before:,}')
print(f'Missing con lai: {int(df.isna().sum().sum()):,}')

So Amount am duoc coi la missing: 0
So dong bi loai vi thieu Class: 0
So gia tri dac trung duoc dien median: 0
Missing con lai: 0


## 3. Xu ly trung lap va danh gia nhieu / gia tri cuc tri

Dong trung lap hoan toan se lam tang trong so cua cung mot diem khi LOF tinh mat do lan can, vi vay can loai bo. Doi voi outlier theo IQR, notebook chi **bao cao** thay vi xoa hoac cat nguong: neu xoa chung, ta co the loai mat giao dich gian lan truoc khi mo hinh duoc hoc.

In [13]:
duplicates_before = int(df.duplicated().sum())
df_clean = df.drop_duplicates().reset_index(drop=True)

def iqr_outlier_count(series: pd.Series) -> pd.Series:
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return pd.Series({
        'lower_bound': lower,
        'upper_bound': upper,
        'iqr_outliers': int(((series < lower) | (series > upper)).sum())
    })

report_cols = [col for col in ['Time', 'Amount'] if col in df_clean.columns]
noise_report = df_clean[report_cols].apply(iqr_outlier_count).T if report_cols else pd.DataFrame()

print(f'Da loai {duplicates_before:,} dong trung lap.')
print(f'Kich thuoc sau lam sach: {df_clean.shape[0]:,} dong x {df_clean.shape[1]} cot')
print('Bao cao gia tri cuc tri theo IQR (chi theo doi, khong xoa):')
display(noise_report)

Da loai 1,081 dong trung lap.
Kich thuoc sau lam sach: 283,726 dong x 31 cot
Bao cao gia tri cuc tri theo IQR (chi theo doi, khong xoa):


,lower_bound,upper_bound,iqr_outliers
Time,-73435.125,266937.875,0.0
Amount,-102.265,185.375,31685.0


## 4. Tach dac trung va chuan hoa du lieu

Theo mo ta bo du lieu, `V1` den `V28` la cac thanh phan da qua bien doi PCA nham bao mat thong tin. Nhom giu nguyen khong gian cac thanh phan nay va chi ap dung `RobustScaler` cho hai cot chua qua PCA la `Time` va `Amount`. `RobustScaler` dua tren median va khoang tu phan vi, nen it bi anh huong boi giao dich co gia tri rat lon.

In [14]:
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL].astype(int)

scale_cols = [col for col in ['Time', 'Amount'] if col in X.columns]
scaler = RobustScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X[scale_cols])

assert X_scaled.isna().sum().sum() == 0, 'X_scaled van con missing values.'
assert np.isfinite(X_scaled.to_numpy()).all(), 'X_scaled van con gia tri khong huu han.'

print(f'X_scaled san sang de lay mau cho LOF: {X_scaled.shape}')
print(f'y dung de lay mau va danh gia: {y.shape}')
display(X_scaled[scale_cols].describe().round(4))

X_scaled san sang de lay mau cho LOF: (283726, 30)
y dung de lay mau va danh gia: (283726,)


,Time,Amount
count,283726.0000,283726.0000
mean,0.1189,0.9244
std,0.5580,3.4821
min,-0.9953,-0.3059
25%,-0.3583,-0.2281
50%,0.0000,0.0000
75%,0.6417,0.7719
max,1.0353,356.9623


## 5. Lay mau va luu tap du lieu cho LOF

LOF can tinh mat do lan can va khoang cach tren du lieu nhieu chieu, nen chay truc tiep tren hang tram nghin giao dich se ton nhieu tai nguyen. Nhom thuc hien **under-sampling** lop binh thuong: giu toan bo giao dich gian lan va lay ngau nhien toi da `N_NORMAL_SAMPLE` giao dich binh thuong. `Class` chi dung de lay mau va danh gia, khong duoc dua vao mo hinh LOF.

In [15]:
df_preprocessed = X_scaled.copy()
df_preprocessed[TARGET_COL] = y.to_numpy()

df_fraud = df_preprocessed[df_preprocessed[TARGET_COL] == 1]
df_normal = df_preprocessed[df_preprocessed[TARGET_COL] == 0]

normal_sample_size = min(N_NORMAL_SAMPLE, len(df_normal))
df_normal_sampled = df_normal.sample(n=normal_sample_size, random_state=RANDOM_STATE)

df_final = pd.concat([df_normal_sampled, df_fraud], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df_final.to_csv(OUTPUT_PATH, index=False)

print(f'So giao dich binh thuong da lay mau: {normal_sample_size:,}')
print(f'So giao dich gian lan duoc giu lai: {len(df_fraud):,}')
print(f'Da luu tap du lieu thu gon tai: {OUTPUT_PATH.resolve()}')
print(f'Kich thuoc file du lieu cho LOF: {df_final.shape}')
display(df_final[TARGET_COL].value_counts().rename('so_luong').to_frame())
display(df_final.head())

So giao dich binh thuong da lay mau: 10,000
So giao dich gian lan duoc giu lai: 473
Da luu tap du lieu thu gon tai: C:\Users\Admin\OneDrive\Desktop\Datamining\creditcard_preprocessed_for_lof.csv
Kich thuoc file du lieu cho LOF: (10473, 31)


,so_luong
0,10000
1,473


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.373702,-1.080483,0.846986,-0.394352,-0.558698,1.590957,1.333946,0.499037,0.873895,-0.633218,-1.695323,-0.018668,0.174666,0.023227,-0.688044,0.889949,-0.912170,1.871919,-1.492511,-1.880440,-0.125842,0.330110,0.905075,-0.100022,-0.729690,-0.051358,0.554370,0.007865,0.046606,0.458907,0
1,0.434153,-0.316294,0.093598,-0.182054,-0.464817,2.900275,4.142079,-0.172407,1.195041,0.278999,-0.707168,-0.757462,0.369412,-0.531936,-0.159520,-1.198724,-0.490853,-0.224625,-0.844563,-0.072429,-0.066127,-0.431264,-1.245281,0.241595,0.608262,-0.377637,-1.130772,0.131027,0.070772,-0.041024,0
2,0.231552,-1.049225,-0.072280,1.566195,-2.871399,-0.901545,-0.831067,-0.516465,0.522248,3.203277,-2.739069,-0.617220,-2.622706,0.524453,1.454934,-0.134486,0.624582,-0.113408,1.151208,-1.072218,-0.317210,0.232372,0.733776,-0.100061,-0.174101,-0.307925,-0.432352,0.014496,0.056275,0.284800,0
3,-0.563059,-0.859577,0.913379,1.634410,-1.352680,-0.023133,-0.515547,0.595465,-0.097411,-0.016912,-0.663406,-0.176540,0.273846,0.452144,-0.187898,0.720024,0.255492,-0.440935,-0.868098,-1.008776,-0.149700,0.054760,-0.016846,-0.046817,0.134409,-0.502241,0.689326,-0.376266,0.102777,-0.207621,0
4,-0.037189,1.334298,-1.552223,0.937684,-1.419113,-1.908205,0.042943,-1.587986,0.018680,-1.263985,1.331458,-1.223005,-0.894342,1.413636,-0.917562,1.174311,0.429097,-0.126896,0.930854,-0.378808,-0.045294,0.083608,0.487591,-0.195867,-0.434731,0.298962,-0.000923,0.068064,0.047571,1.209846,0
